In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics piexif lap

In [ ]:
# Import necessary libraries
import cv2
import os
import torch
# from google.colab.patches import cv2_imshow # Not needed for this task
import time
# Import YOLO directly from ultralytics
from ultralytics import YOLO
from PIL import Image # Import Pillow for image manipulation
import piexif # Import piexif for EXIF data handling
import numpy as np  # Added numpy import for array operations
from collections import defaultdict, deque
import zipfile  # Added for creating zip archives
import shutil  # Added for file operations

# --- Installation Step ---
# This cell installs the necessary 'ultralytics' package required by YOLO.
# Run this cell first if it hasn't been run.
!pip install ultralytics piexif -q # Added piexif for EXIF handling

# --- Configuration ---
# Define the base directory for input video files
INPUT_BASE_DIR = '/content/drive/MyDrive/Crab behaviour analysis_UniMAP & NSTRU/Video/Dataset/' # Updated base directory

# Define the base directory for output processed frames
OUTPUT_BASE_DIR = '/content/drive/MyDrive/Crab behaviour analysis_UniMAP & NSTRU/Video/Processed_dataset/' # Updated base directory

# Define the path to your custom YOLOv8 weights file
CUSTOM_WEIGHTS_PATH = '/content/drive/MyDrive/Crab behaviour analysis_UniMAP & NSTRU/weights-yolov11-model-53.pt'

# Preprocessing settings
TARGET_FPS = 24  # Extract 24 frames per second - Changed from 5
RESIZE_DIM = (224, 224) # Target size for the behavior classification model (width, height)
PREDICTION_SIZE = (640, 640) # Size for YOLO prediction preprocessing (width, height) - ADDED
CONFIDENCE_THRESHOLD = 0.5 # Only detect objects with a confidence score higher than this
OVERLAP_THRESHOLD = 0.6 # IoU threshold for filtering overlapping detections
PADDING_PERCENTAGE = 0.1 # Percentage of bounding box size to add as padding
USE_GRAYSCALE = True # Flag to enable grayscale preprocessing for YOLO prediction - ADDED
SAVE_CROPPED_GRAYSCALE = True # Flag to save cropped images as grayscale - ADDED

# Tracking settings for built-in tracker
TRACKER_CONFIG = 'botsort.yaml'  # Use ByteTrack (can also be 'botsort.yaml')
PERSIST_TRACKS = True  # Persist tracks between frames
INTERPOLATE_MISSING = True  # Interpolate missing detections

# --- Load Custom YOLO Model ---
yolo_model_custom = None # Initialize to None
class_names_custom = None # Initialize to None

try:
    print(f"Attempting to load custom YOLO model using ultralytics.YOLO from: {CUSTOM_WEIGHTS_PATH}")
    # Assuming it's a YOLOv8 model based on user request
    yolo_model_custom = YOLO(CUSTOM_WEIGHTS_PATH) # Load the detection model first
    print(f"Custom YOLO model loaded successfully from {CUSTOM_WEIGHTS_PATH}.")

    # Get class names from the loaded model
    if hasattr(yolo_model_custom, 'names'):
         class_names_custom = yolo_model_custom.names
         print(f"Loaded {len(class_names_custom)} custom YOLO class names: {class_names_custom}")
         # Verify if 'fiddler_crab' is in class_names if that's the expected class
         if 'fiddler_crab' in class_names_custom.values(): # Check values as names is a dict in newer ultralytics
             print("Confirmed 'fiddler_crab' class is in the custom model's class names.")
         else:
             print("Warning: 'fiddler_crab' class not found in the custom model's class names.")
    else:
        class_names_custom = None
        print("Could not load custom YOLO class names from the model.")

except Exception as e:
    print(f"Error loading custom YOLO model from {CUSTOM_WEIGHTS_PATH} using ultralytics.YOLO: {e}")
    print("Failed to load the custom YOLOv8 model. Cannot proceed with processing.")
    yolo_model_custom = None
    class_names_custom = None


# --- Helper Classes for Track Management ---
class TrackManager:
    """Manages track continuity and interpolation for missing frames."""

    def __init__(self, max_missing_frames=5):
        self.tracks = {}  # track_id -> track_info
        self.max_missing_frames = max_missing_frames
        self.frame_count = 0

    def update_tracks(self, current_tracks):
        """Update tracks with current frame detections."""
        self.frame_count += 1
        current_track_ids = set()

        # Handle the case where current_tracks is a Boxes object
        if current_tracks is not None and hasattr(current_tracks, 'id') and current_tracks.id is not None:
            # Extract data from Boxes object
            track_ids = current_tracks.id.cpu().numpy().astype(int)
            bboxes = current_tracks.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
            confidences = current_tracks.conf.cpu().numpy()
            classes = current_tracks.cls.cpu().numpy().astype(int)

            # Process each detection
            for i, track_id in enumerate(track_ids):
                if track_id is not None:
                    current_track_ids.add(track_id)

                    bbox = bboxes[i]
                    conf = float(confidences[i])
                    cls = int(classes[i])

                    if track_id not in self.tracks:
                        # New track
                        self.tracks[track_id] = {
                            'id': track_id,
                            'history': deque(maxlen=10),
                            'last_seen': self.frame_count,
                            'confidence': conf,
                            'class_id': cls,
                            'active': True
                        }

                    # Update track info
                    self.tracks[track_id]['last_seen'] = self.frame_count
                    self.tracks[track_id]['confidence'] = conf
                    self.tracks[track_id]['active'] = True
                    self.tracks[track_id]['history'].append({
                        'frame': self.frame_count,
                        'bbox': bbox,
                        'confidence': conf
                    })

        # Mark missing tracks as inactive
        for track_id in self.tracks:
            if track_id not in current_track_ids:
                self.tracks[track_id]['active'] = False

        return self.get_all_tracks()

    def get_all_tracks(self):
        """Get all tracks including interpolated ones for missing subjects."""
        all_tracks = []

        for track_id, track_info in self.tracks.items():
            frames_since_seen = self.frame_count - track_info['last_seen']

            if track_info['active']:
                # Active track - use latest detection
                if track_info['history']:
                    latest = track_info['history'][-1]
                    all_tracks.append({
                        'track_id': track_id,
                        'bbox': latest['bbox'],
                        'confidence': latest['confidence'],
                        'class_id': track_info['class_id'],
                        'state': 'tracked'
                    })
            elif frames_since_seen <= self.max_missing_frames and len(track_info['history']) >= 2:
                # Interpolate missing track
                interpolated_bbox = self.interpolate_position(track_info, frames_since_seen)
                interpolated_conf = max(0.1, track_info['confidence'] - 0.1 * frames_since_seen)

                all_tracks.append({
                    'track_id': track_id,
                    'bbox': interpolated_bbox,
                    'confidence': interpolated_conf,
                    'class_id': track_info['class_id'],
                    'state': 'interpolated'
                })

        # Ensure we always have at least one track (use best available)
        if not all_tracks and self.tracks:
            best_track = max(self.tracks.values(), key=lambda x: x['confidence'])
            if best_track['history']:
                latest = best_track['history'][-1]
                all_tracks.append({
                    'track_id': best_track['id'],
                    'bbox': latest['bbox'],
                    'confidence': max(0.05, best_track['confidence'] - 0.2),
                    'class_id': best_track['class_id'],
                    'state': 'recovered'
                })

        return all_tracks

    def interpolate_position(self, track_info, frames_missing):
        """Interpolate position based on recent movement."""
        if len(track_info['history']) < 2:
            return track_info['history'][-1]['bbox']

        # Use last two positions to calculate velocity
        last_pos = track_info['history'][-1]['bbox']
        prev_pos = track_info['history'][-2]['bbox']

        # Calculate velocity (change per frame)
        velocity = (last_pos - prev_pos)

        # Extrapolate position
        interpolated_pos = last_pos + velocity * frames_missing

        return interpolated_pos


# --- Helper function to apply EXIF orientation ---
def apply_exif_orientation(image_path, image):
    """
    Adds EXIF orientation handling to the image loading process.
    Returns the correctly oriented image (as a NumPy array).
    """
    try:
        # Open image with Pillow to read EXIF data
        pil_img = Image.open(image_path)

        if "_getexif" in dir(pil_img) and pil_img._getexif():
            exif_dict = pil_img._getexif()
            if exif_dict:
                # 0x0112 is the tag for Orientation
                orientation = exif_dict.get(0x0112)

                if orientation:
                    # Convert back to OpenCV format (BGR) before applying rotations
                    if pil_img.mode == 'RGB':
                        cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
                    elif pil_img.mode == 'L': # Grayscale
                         cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_GRAY2BGR) # Convert grayscale to BGR
                    else:
                         cv_img = np.array(pil_img) # Keep original if not RGB or L

                    if orientation == 2:
                        cv_img = cv2.flip(cv_img, 1) # Flip horizontal
                    elif orientation == 3:
                        cv_img = cv2.rotate(cv_img, cv2.ROTATE_180) # Rotate 180
                    elif orientation == 4:
                        cv_img = cv2.flip(cv_img, 0) # Flip vertical
                    elif orientation == 5:
                        cv_img = cv2.rotate(cv_img, cv2.ROTATE_90_CLOCKWISE) # Rotate 90 CW
                        cv_img = cv2.flip(cv_img, 1) # Flip horizontal
                    elif orientation == 6:
                        cv_img = cv2.rotate(cv_img, cv2.ROTATE_90_CLOCKWISE) # Rotate 90 CW
                    elif orientation == 7:
                        cv_img = cv2.rotate(cv2.ROTATE_90_COUNTERCLOCKWISE) # Rotate 90 CCW
                        cv_img = cv2.flip(cv_img, 1) # Flip horizontal
                    elif orientation == 8:
                        cv_img = cv2.rotate(cv_img, cv2.ROTATE_90_COUNTERCLOCKWISE) # Rotate 90 CCW

                    return cv_img # Return oriented OpenCV image

        # If no EXIF data or orientation, return the original OpenCV image
        return image

    except Exception as e:
        # Handle potential errors during EXIF reading
        # print(f"Warning: Could not read or apply EXIF orientation for {image_path}: {e}") # Commented out to reduce log spam
        return image # Return original image if EXIF handling fails


# --- NEW HELPER FUNCTION: Preprocess frame for YOLO prediction ---
def preprocess_frame_for_prediction(frame, target_size=(640, 640), use_grayscale=True):
    """
    Preprocesses a frame for YOLO prediction by resizing to target size and optionally converting to grayscale.

    Args:
        frame: Input frame (BGR format)
        target_size: Tuple (width, height) for resizing
        use_grayscale: Boolean flag to convert to grayscale

    Returns:
        Preprocessed frame and scale factors for bbox coordinate adjustment
    """
    original_height, original_width = frame.shape[:2]
    target_width, target_height = target_size

    # Calculate scale factors for later bbox coordinate adjustment
    scale_x = original_width / target_width
    scale_y = original_height / target_height

    # Resize frame to target size
    resized_frame = cv2.resize(frame, target_size, interpolation=cv2.INTER_LINEAR)

    # Convert to grayscale if requested
    if use_grayscale:
        # Convert BGR to grayscale
        gray_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
        # Convert back to 3-channel for YOLO compatibility
        preprocessed_frame = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2BGR)
    else:
        preprocessed_frame = resized_frame

    return preprocessed_frame, scale_x, scale_y


# --- NEW HELPER FUNCTION: Adjust bbox coordinates ---
def adjust_bbox_coordinates(bbox, scale_x, scale_y):
    """
    Adjusts bounding box coordinates from preprocessed frame back to original frame coordinates.

    Args:
        bbox: Bounding box [xmin, ymin, xmax, ymax] in preprocessed frame coordinates
        scale_x: Scale factor for x-axis
        scale_y: Scale factor for y-axis

    Returns:
        Adjusted bounding box in original frame coordinates
    """
    xmin, ymin, xmax, ymax = bbox
    adjusted_bbox = [
        xmin * scale_x,
        ymin * scale_y,
        xmax * scale_x,
        ymax * scale_y
    ]
    return adjusted_bbox


# --- SIMPLIFIED Helper function using model.track() ---
def process_frame_with_tracking(frame, image_path, model, names, track_manager, output_folder_path, video_name_without_ext, frame_count, confidence_threshold, padding_percentage):
    """
    Processes a single frame using YOLO's built-in tracking (model.track()),
    saves cropped images of tracked objects, and returns information.
    Much simpler than custom ByteTrack implementation!
    """
    tracked_objects_info = []

    if model is None:
        return tracked_objects_info

    # Apply EXIF orientation
    oriented_frame = apply_exif_orientation(image_path, frame)

    # Preprocess frame for YOLO prediction (resize to 640x640 and optionally convert to grayscale)
    preprocessed_frame, scale_x, scale_y = preprocess_frame_for_prediction(
        oriented_frame,
        target_size=PREDICTION_SIZE,
        use_grayscale=USE_GRAYSCALE
    )

    # Use YOLO's built-in tracking - much simpler!
    results = model.track(
        preprocessed_frame,
        conf=confidence_threshold,
        tracker=TRACKER_CONFIG,  # Use ByteTrack
        persist=PERSIST_TRACKS,  # Persist tracks between frames
        verbose=False
    )

    # Extract tracking results
    current_tracks = None
    if results and len(results) > 0 and results[0].boxes is not None:
        if hasattr(results[0].boxes, 'id') and results[0].boxes.id is not None:
            current_tracks = results[0].boxes

    # Update track manager with current detections
    all_tracks = track_manager.update_tracks(current_tracks)

    # Find the highest confidence track for this frame
    best_track = None
    if all_tracks:
        best_track = max(all_tracks, key=lambda x: x['confidence'])

    # Process only the highest confidence track
    if best_track:
        track_id = best_track['track_id']
        bbox = best_track['bbox']  # [x1, y1, x2, y2] format
        confidence = best_track['confidence']
        class_id = best_track['class_id']
        state = best_track['state']  # 'tracked', 'interpolated', or 'recovered'

        # Adjust bbox coordinates back to original frame size
        adjusted_bbox = adjust_bbox_coordinates(bbox, scale_x, scale_y)
        xmin, ymin, xmax, ymax = [int(coord) for coord in adjusted_bbox]

        class_name = names.get(class_id, f"Class {class_id}") if names else f"Class {class_id}"

        # Calculate padding
        bbox_width = xmax - xmin
        bbox_height = ymax - ymin
        pad_x = int(bbox_width * padding_percentage)
        pad_y = int(bbox_height * padding_percentage)

        # Apply padding and ensure coordinates are within frame bounds
        h, w, _ = oriented_frame.shape
        crop_xmin = max(0, xmin - pad_x)
        crop_ymin = max(0, ymin - pad_y)
        crop_xmax = min(w, xmax + pad_x)
        crop_ymax = min(h, ymax + pad_y)

        # Crop from the original oriented frame
        cropped_object = oriented_frame[crop_ymin:crop_ymax, crop_xmin:crop_xmax]

        if cropped_object.size > 0 and cropped_object.shape[0] > 0 and cropped_object.shape[1] > 0:
            # Resize the cropped image
            resized_cropped_object = cv2.resize(cropped_object, RESIZE_DIM, interpolation=cv2.INTER_AREA)

            # Convert to grayscale if requested
            if SAVE_CROPPED_GRAYSCALE:
                resized_cropped_object = cv2.cvtColor(resized_cropped_object, cv2.COLOR_BGR2GRAY)

            # Save the cropped image with track ID and state information
            state_suffix = f"_{state}" if state != "tracked" else ""
            output_filename = f"{video_name_without_ext}_frame{frame_count:05d}_track{track_id}_conf{confidence:.4f}{state_suffix}.jpg"
            output_path = os.path.join(output_folder_path, output_filename)
            cv2.imwrite(output_path, resized_cropped_object)

            # Store information
            tracked_objects_info.append({
                'frame_count': frame_count,
                'track_id': track_id,
                'class_id': class_id,
                'class_name': class_name,
                'confidence': confidence,
                'state': state,
                'xmin': crop_xmin, 'ymin': crop_ymin, 'xmax': crop_xmax, 'ymax': crop_ymax,
                'cropped_image_path': output_path
            })
        else:
            print(f"Warning: Empty or invalid crop for track {track_id} at frame {frame_count}. Skipping saving.")

    return tracked_objects_info


# --- Helper function to create zip archive ---
def create_zip_archive(source_folder, output_path, exclude_patterns=None):
    """
    Creates a zip archive of the source folder.

    Args:
        source_folder: Path to the folder to zip
        output_path: Path where the zip file will be created
        exclude_patterns: List of patterns to exclude from the zip
    """
    if exclude_patterns is None:
        exclude_patterns = []

    print(f"Creating zip archive: {output_path}")
    print(f"Source folder: {source_folder}")

    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        total_files = 0
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                # Check if file should be excluded
                should_exclude = False
                for pattern in exclude_patterns:
                    if pattern in file or pattern in root:
                        should_exclude = True
                        break

                if not should_exclude:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, source_folder)
                    zipf.write(file_path, arcname)
                    total_files += 1

                    if total_files % 1000 == 0:  # Progress indicator
                        print(f"  Added {total_files} files to zip...")

    # Get zip file size
    zip_size = os.path.getsize(output_path)
    zip_size_mb = zip_size / (1024 * 1024)

    print(f"Zip archive created successfully!")
    print(f"  Total files: {total_files}")
    print(f"  Archive size: {zip_size_mb:.2f} MB")

    return output_path


# --- Main Video Processing Loop ---

print(f"\n--- Starting video processing with YOLO built-in tracking from: {INPUT_BASE_DIR} ---")
print(f"Preprocessing settings: Resize to {PREDICTION_SIZE}, Grayscale: {USE_GRAYSCALE}")
print(f"Output settings: Save cropped images as grayscale: {SAVE_CROPPED_GRAYSCALE}")
print(f"Tracking settings: {TRACKER_CONFIG}, Persist: {PERSIST_TRACKS}")

# Initialize the list to store tracked objects information for all videos
all_tracked_objects_info = []

# Check if the input base directory exists
if not os.path.isdir(INPUT_BASE_DIR):
    print(f"Error: Input base directory not found at {INPUT_BASE_DIR}")
else:
    # Iterate through gender directories
    for gender_name in os.listdir(INPUT_BASE_DIR):
        gender_dir = os.path.join(INPUT_BASE_DIR, gender_name)
        if not os.path.isdir(gender_dir):
            continue

        print(f"\nProcessing gender directory: {gender_name}")

        # Iterate through activity directories within the gender directory
        for activity_name in os.listdir(gender_dir):
            activity_dir = os.path.join(gender_dir, activity_name)
            if not os.path.isdir(activity_dir):
                continue

            print(f"  Processing activity directory: {activity_name}")

            # Iterate through video files within the activity directory
            for filename in os.listdir(activity_dir):
                # Check if the file is a video (you might want to add more video extensions)
                if filename.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                    video_file_path = os.path.join(activity_dir, filename)
                    video_name_without_ext = os.path.splitext(filename)[0]

                    print(f"\n    Processing video file: {filename}")

                    # Define the output folder for this video, maintaining gender/activity structure
                    # Output structure: OUTPUT_BASE_DIR / gender_name / activity_name / video_name /
                    video_output_folder = os.path.join(OUTPUT_BASE_DIR, gender_name, activity_name, video_name_without_ext)

                    # Create the output folder if it doesn't exist
                    if not os.path.exists(video_output_folder):
                        os.makedirs(video_output_folder)
                        print(f"    Created output directory: {video_output_folder}")
                    else:
                         print(f"    Output directory already exists: {video_output_folder}. Content will be added.")

                    # Initialize track manager for this video
                    track_manager = TrackManager(max_missing_frames=5)

                    # Open the video file
                    cap = cv2.VideoCapture(video_file_path)
                    if not cap.isOpened():
                        print(f"    Error: Could not open video {filename}. Skipping.")
                        continue # Skip to the next file

                    # Get video properties
                    original_fps = cap.get(cv2.CAP_PROP_FPS)
                    frame_count_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

                    if original_fps <= 0:
                        print(f"    Warning: Video {filename} has an invalid FPS ({original_fps}). Cannot determine duration. Defaulting to frame interval 1.")
                        frame_interval = 1
                    else:
                         # Calculate frame interval based on target FPS
                         frame_interval = max(1, int(original_fps / TARGET_FPS))
                         print(f"    Original FPS: {original_fps}, Target FPS: {TARGET_FPS}, Frame interval: {frame_interval}")

                    frame_count = 0
                    processed_frame_count = 0

                    while True:
                        ret, frame = cap.read()
                        if not ret:
                            break

                        # Only process frames at the calculated interval
                        if frame_count % frame_interval == 0:
                            # Process frame with YOLO built-in tracking
                            if yolo_model_custom:
                                frame_tracked_objects = process_frame_with_tracking(
                                    frame,
                                    video_file_path, # Pass video path for EXIF
                                    yolo_model_custom,
                                    class_names_custom,
                                    track_manager,
                                    video_output_folder, # Save to the video-specific output folder
                                    video_name_without_ext,
                                    processed_frame_count,
                                    CONFIDENCE_THRESHOLD,
                                    PADDING_PERCENTAGE
                                )
                                all_tracked_objects_info.extend(frame_tracked_objects)

                                if processed_frame_count % 100 == 0:  # Progress indicator
                                    print(f"      Processed {processed_frame_count} frames, found {len(frame_tracked_objects)} tracks in current frame")
                            else:
                                pass # Do nothing if model is not loaded

                            processed_frame_count += 1

                        frame_count += 1

                    cap.release()
                    print(f"    Finished processing video: {filename}. Total processed frames: {processed_frame_count}")
                    print(f"    Saved cropped images with YOLO tracking to {video_output_folder}.")

    print("\n--- Finished processing all video files with YOLO built-in tracking. ---")

    # Summary of tracking results
    print("\n--- Summary of YOLO Tracking Results ---")
    if all_tracked_objects_info:
        print(f"Total tracked object entries across all processed videos: {len(all_tracked_objects_info)}")

        # Count by tracking state
        tracked_count = len([obj for obj in all_tracked_objects_info if obj['state'] == 'tracked'])
        interpolated_count = len([obj for obj in all_tracked_objects_info if obj['state'] == 'interpolated'])
        recovered_count = len([obj for obj in all_tracked_objects_info if obj['state'] == 'recovered'])

        print(f"  - Direct detections: {tracked_count}")
        print(f"  - Interpolated tracks: {interpolated_count}")
        print(f"  - Recovered tracks: {recovered_count}")

        # Count unique tracks
        unique_tracks = set(obj['track_id'] for obj in all_tracked_objects_info)
        print(f"  - Unique track IDs: {len(unique_tracks)}")

        # Print examples
        print("Examples:")
        for i, obj_info in enumerate(all_tracked_objects_info):
             video_name = os.path.basename(os.path.dirname(obj_info['cropped_image_path']))
             state_info = f" [{obj_info['state']}]" if obj_info['state'] != 'tracked' else ""
             print(f"  {i+1}: Video: {video_name}, Frame: {obj_info['frame_count']}, Track: {obj_info['track_id']}, "
                   f"Class: {obj_info['class_name']}, Confidence: {obj_info['confidence']:.2f}{state_info}")
             if i >= 9: # Print only first 10 examples
                 break
    else:
        print("No tracked objects were found across all processed videos.")

    # Save tracking results to CSV for further analysis
    csv_path = None
    if all_tracked_objects_info:
        import csv
        csv_path = os.path.join(OUTPUT_BASE_DIR, 'yolo_tracking_results.csv')
        with open(csv_path, 'w', newline='') as csvfile:
            fieldnames = ['video_path', 'frame_count', 'track_id', 'class_name', 'confidence', 'state', 'xmin', 'ymin', 'xmax', 'ymax', 'cropped_image_path']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for obj in all_tracked_objects_info:
                writer.writerow({
                    'video_path': os.path.dirname(obj['cropped_image_path']),
                    'frame_count': obj['frame_count'],
                    'track_id': obj['track_id'],
                    'class_name': obj['class_name'],
                    'confidence': obj['confidence'],
                    'state': obj['state'],
                    'xmin': obj['xmin'],
                    'ymin': obj['ymin'],
                    'xmax': obj['xmax'],
                    'ymax': obj['ymax'],
                    'cropped_image_path': obj['cropped_image_path']
                })
        print(f"\n--- Tracking results saved to: {csv_path} ---")
    else:
        print("\n--- No tracking data to save to CSV ---")

    # Create zip archive of all processed data
    print("\n--- Creating zip archive of processed data ---")

    # Generate timestamp for unique zip filename
    zip_filename = f"crab_behavior_processed_data_v2.zip"
    zip_path = os.path.join(OUTPUT_BASE_DIR, zip_filename)

    try:
        # Create zip archive of the entire processed dataset
        create_zip_archive(
            source_folder=OUTPUT_BASE_DIR,
            output_path=zip_path,
            exclude_patterns=['.zip']  # Exclude any existing zip files to avoid recursion
        )

        print(f"\n--- Archive created successfully: {zip_path} ---")

        # Calculate some final statistics
        if os.path.exists(zip_path):
            zip_size = os.path.getsize(zip_path)
            zip_size_mb = zip_size / (1024 * 1024)
            zip_size_gb = zip_size_mb / 1024

            print(f"Final archive statistics:")
            print(f"  - Archive size: {zip_size_mb:.2f} MB ({zip_size_gb:.2f} GB)")
            print(f"  - Total processed images: {len(all_tracked_objects_info)}")
            print(f"  - Archive location: {zip_path}")

        print(f"\n--- Processing complete! ---")
        print(f"All processed data has been saved and archived.")

    except Exception as e:
        print(f"Error creating zip archive: {e}")
        print("Processing completed but archive creation failed.")

print(f"\n--- Script execution completed ---")

In [ ]:
import os

OUTPUT_BASE_DIR = '/content/drive/MyDrive/Crab behaviour analysis_UniMAP & NSTRU/Video/Processed_dataset/'

empty_dirs = []

# Walk through everything inside OUTPUT_BASE_DIR
for root, dirs, files in os.walk(OUTPUT_BASE_DIR):
    # A directory is empty if it has no files and no subdirectories
    if not dirs and not files:
        empty_dirs.append(root)

print("Empty directories found:")
for d in empty_dirs:
    print(d)

print(f"\nTotal empty dirs: {len(empty_dirs)}")


In [ ]:
import zipfile  # Added for creating zip archives
import os
import shutil

ZIP_FILE_PATH = '/content/drive/MyDrive/Crab behaviour analysis_UniMAP & NSTRU/Video/Processed_dataset/crab_behavior_processed_data_waving_feeding_no_others_QC.zip'

LOCAL_EXTRACT_PATH = '/content/crab_behavior_data'  # Local path to extract data
DATASET_BASE_PATH = os.path.join(LOCAL_EXTRACT_PATH, 'Processed_dataset')  # Adjust based on zip structure

def setup_local_dataset(zip_path, extract_path):
    """
    Download and extract the crab behavior dataset locally

    Args:
        zip_path: Path to the zip file in Google Drive
        extract_path: Local path where to extract the data

    Returns:
        str: Path to the extracted dataset
    """
    print(f"Setting up local dataset...")
    print(f"Source zip file: {zip_path}")
    print(f"Local extraction path: {extract_path}")

    # Check if zip file exists
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip file not found at {zip_path}")

    # Remove existing extraction directory if it exists
    if os.path.exists(extract_path):
        print(f"Removing existing directory: {extract_path}")
        shutil.rmtree(extract_path)

    # Create extraction directory
    os.makedirs(extract_path, exist_ok=True)

    # Extract zip file
    print(f"Extracting zip file...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"Successfully extracted zip file to {extract_path}")

        # List contents of extracted directory to understand structure
        print(f"\nContents of {extract_path}:")
        for item in os.listdir(extract_path):
            item_path = os.path.join(extract_path, item)
            if os.path.isdir(item_path):
                print(f"  Directory: {item}")
                # List subdirectories
                try:
                    subdirs = os.listdir(item_path)
                    for subdir in subdirs[:5]:  # Show first 5 subdirectories
                        print(f"    - {subdir}")
                    if len(subdirs) > 5:
                        print(f"    ... and {len(subdirs) - 5} more")
                except Exception as e:
                    print(f"    Error listing subdirectories: {e}")
            else:
                print(f"  File: {item}")

        return extract_path

    except zipfile.BadZipFile:
        raise ValueError(f"Invalid or corrupted zip file: {zip_path}")
    except Exception as e:
        raise RuntimeError(f"Error extracting zip file: {e}")

# Function to find the correct dataset path after extraction
def find_dataset_path(extract_path):
    """
    Find the actual dataset path after extraction (handles different zip structures)

    Args:
        extract_path: Base extraction path

    Returns:
        str: Path to the dataset directory containing gender/activity folders
    """
    # Common possible paths after extraction
    possible_paths = [
        extract_path,
        os.path.join(extract_path, 'Processed_dataset'),
        os.path.join(extract_path, 'crab_behavior_processed_data'),
        os.path.join(extract_path, 'Video', 'Processed_dataset'),
    ]

    # Check for subdirectories that might contain the actual dataset
    for root, dirs, files in os.walk(extract_path):
        if any(d.lower() in ['male', 'female', 'gender_0', 'gender_1'] for d in dirs):
            possible_paths.append(root)

    # Test each possible path
    for path in possible_paths:
        if os.path.exists(path):
            subdirs = os.listdir(path) if os.path.isdir(path) else []
            # Check if this looks like a dataset directory (contains gender or activity folders)
            if any(d.lower() in ['male', 'female'] for d in subdirs) or len(subdirs) > 0:
                print(f"Found dataset at: {path}")
                print(f"Contains directories: {subdirs}")
                return path

    # If no suitable path found, return the base extraction path
    print(f"Warning: Could not find a clear dataset structure. Using base path: {extract_path}")
    return extract_path

try:
    local_extract_path = setup_local_dataset(ZIP_FILE_PATH, LOCAL_EXTRACT_PATH)
    DATASET_BASE_PATH = find_dataset_path(local_extract_path)
    print(f"\nFinal dataset path: {DATASET_BASE_PATH}")
except Exception as e:
    print(f"Error setting up local dataset: {e}")
    print("Please check the zip file path and try again.")
    # Exit or use a fallback path
    DATASET_BASE_PATH = '/content/fallback_dataset'  # This will likely cause the script to find no data

# --- Data Loading and Preprocessing ---
sequence_info = []
activity_labels = {}
activity_counter = 0
gender_labels = {}
gender_counter = 0

# Counters for sequences per category
activity_sequence_counts = {}
gender_sequence_counts = {}
gender_activity_sequence_counts = {}

print(f"\nCollecting sequence information from: {DATASET_BASE_PATH}")
print("Assuming structure: gender/activity/sequence/image_files...")

if os.path.exists(DATASET_BASE_PATH) and os.listdir(DATASET_BASE_PATH):
    for gender_name in sorted(os.listdir(DATASET_BASE_PATH)):
        gender_dir = os.path.join(DATASET_BASE_PATH, gender_name)
        if not os.path.isdir(gender_dir):
            continue

        if gender_name not in gender_labels:
            gender_labels[gender_name] = gender_counter
            gender_counter += 1
        current_gender_label = gender_labels[gender_name]

        # Initialize gender count
        if gender_name not in gender_sequence_counts:
            gender_sequence_counts[gender_name] = 0
        if gender_name not in gender_activity_sequence_counts:
            gender_activity_sequence_counts[gender_name] = {}

        for activity_name in sorted(os.listdir(gender_dir)):
            activity_dir = os.path.join(gender_dir, activity_name)
            if not os.path.isdir(activity_dir):
                continue

            if activity_name not in activity_labels:
                activity_labels[activity_name] = activity_counter
                activity_counter += 1
            current_activity_label = activity_labels[activity_name]

            # Initialize activity count
            if activity_name not in activity_sequence_counts:
                activity_sequence_counts[activity_name] = 0
            if activity_name not in gender_activity_sequence_counts[gender_name]:
                gender_activity_sequence_counts[gender_name][activity_name] = 0

            for sequence_name in os.listdir(activity_dir):
                sequence_dir = os.path.join(activity_dir, sequence_name)
                if not os.path.isdir(sequence_dir):
                    continue

                image_files = sorted([f for f in os.listdir(sequence_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

                if not image_files:
                    continue

                image_paths = [os.path.join(sequence_dir, f) for f in image_files]
                sequence_info.append((image_paths, current_activity_label, current_gender_label))

                # Increment counters
                activity_sequence_counts[activity_name] += 1
                gender_sequence_counts[gender_name] += 1
                gender_activity_sequence_counts[gender_name][activity_name] += 1

else:
    print(f"Error: Dataset base path not found or is empty at {DATASET_BASE_PATH}")

print("\n--- Sequence Information Collection Complete ---")
print(f"Total sequences found: {len(sequence_info)}")
print(f"Activity labels mapping: {activity_labels}")
print(f"Gender labels mapping: {gender_labels}")

# Display sequence counts per activity
print("\n--- Sequences per Activity ---")
for activity, count in sorted(activity_sequence_counts.items()):
    print(f"{activity}: {count} sequences")

# Display sequence counts per gender
print("\n--- Sequences per Gender ---")
for gender, count in sorted(gender_sequence_counts.items()):
    print(f"{gender}: {count} sequences")

# Display detailed breakdown: sequences per activity within each gender
print("\n--- Detailed Breakdown: Sequences per Gender-Activity Combination ---")
for gender in sorted(gender_activity_sequence_counts.keys()):
    print(f"\n{gender}:")
    total_gender_sequences = 0
    for activity in sorted(gender_activity_sequence_counts[gender].keys()):
        count = gender_activity_sequence_counts[gender][activity]
        total_gender_sequences += count
        print(f"  {activity}: {count} sequences")
    print(f"  Total for {gender}: {total_gender_sequences} sequences")

num_activity_classes = len(activity_labels) if activity_labels else 1
num_gender_classes = len(gender_labels) if gender_labels else 1

# Extract paths and labels for augmentation and splitting
all_image_paths = [item[0] for item in sequence_info]
all_activity_labels = [item[1] for item in sequence_info]
all_gender_labels = [item[2] for item in sequence_info]